In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Clone repo into Drive (one time only)
repo_url = "https://github.com/derekn4/llm-training-pipeline.git"
clone_path = "/content/drive/MyDrive/llm-training-pipeline"

if not os.path.exists(clone_path):
    !git clone {repo_url} {clone_path}
    print("Repo cloned ✓")
else:
    print("Repo already exists, pulling latest...")
    !git -C {clone_path} pull

os.chdir(clone_path)
print(f"Working directory: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already exists, pulling latest...
fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
Working directory: /content/drive/MyDrive/llm-training-pipeline


In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/llm-training-pipeline')

# Pull latest changes from GitHub
!git pull

print(f"Working directory: {os.getcwd()}")
!ls

Mounted at /content/drive
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.63 KiB | 24.00 KiB/s, done.
From https://github.com/derekn4/llm-training-pipeline
   89c572d..82b626f  main       -> origin/main
Updating 89c572d..82b626f
Fast-forward
 00_setup.ipynb   | 119 ++++++++++++++++++++++++++++++++++++++++++++++++++++++-
 requirements.txt |   8 ++++
 2 files changed, 126 insertions(+), 1 deletion(-)
Working directory: /content/drive/MyDrive/llm-training-pipeline
00_setup.ipynb	   02_sft.ipynb  04_eval.ipynb	requirements.txt
01_pretrain.ipynb  03_dpo.ipynb  README.md


In [6]:
!nvidia-smi

Sun Jul 12 23:15:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 23.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 122.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 21.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 10.2 MB/s eta 0:00:00


In [1]:
!pip install -q python-dotenv

In [ ]:
import torch
from google.colab import userdata
import wandb
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from dotenv import load_dotenv
import os

load_dotenv('/content/drive/MyDrive/llm-training-pipeline/.env')

# Login to Hugging Face Hub
from huggingface_hub import login
login(token=os.getenv("HF_API_KEY"))

# Login to wandb
wandb.login(key=os.getenv("WANDB_API_KEY"))
wandb.init(project="llm-training-pipeline", name="smoke-test")

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # use 4-bit instead of 16/32-bit
    bnb_4bit_quant_type="nf4",      # NormalFloat4: best quality 4-bit format
    bnb_4bit_compute_dtype=torch.float16,  # upcast to fp16 during actual math
    bnb_4bit_use_double_quant=True, # quantize the quantization constants too, saves ~0.4GB extra
)
# Load Qwen 2.5 1.5B
model_id = "Qwen/Qwen2.5-1.5B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded ✓")

# Generate a completion
prompt = "Explain what a transformer neural network is in simple terms:"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n--- Generation ---")
print(response)

# Log to wandb
wandb.log({
    "prompt": prompt,
    "response": response,
    "model": model_id,
    "quantization": "4-bit NF4",
    "gpu": torch.cuda.get_device_name(0),
    "vram_used_gb": torch.cuda.memory_allocated() / 1e9,
})

print("\nLogged to W&B ✓")
wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Loading tokenizer...
Loading model in 4-bit...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Model loaded ✓

--- Generation ---
Explain what a transformer neural network is in simple terms: A transformer neural network is like a special machine that helps computers understand and process information better by learning how to "transform" or rearrange the information it is given. It does this by using a series of steps called layers, similar to how water flows through a pipe. This helps the computer make sense of different inputs and generate better outputs.

Logged to W&B ✓


vram_used_gb,▁
gpu,Tesla T4
model,Qwen/Qwen2.5-1.5B
prompt,Explain what a trans...
quantization,4-bit NF4
response,Explain what a trans...
vram_used_gb,1.84904


In [1]:
from dotenv import load_dotenv
from google.colab import drive
import os

drive.mount('/content/drive')
load_dotenv('/content/drive/MyDrive/llm-training-pipeline/.env')
os.chdir('/content/drive/MyDrive/llm-training-pipeline')

# Pull latest from GitHub
!git pull

Mounted at /content/drive
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 4.84 KiB | 105.00 KiB/s, done.
From https://github.com/derekn4/llm-training-pipeline
   82b626f..2b98fc4  main       -> origin/main
Updating 82b626f..2b98fc4
Fast-forward
 .gitignore     |   1 +
 00_setup.ipynb | 370 ++++++++++++++++++++++++++++++++++++++++++++++++++++++++-
 2 files changed, 365 insertions(+), 6 deletions(-)
 create mode 100644 .gitignore
